In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    classification_report,
)

pd.set_option("display.max_columns", None)

In [2]:
CSV_PATH = "../../data/forecast_vs_actual.csv"
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
df.head()

Shape: (4800, 23)


,forecast_id,station_id,target_timestamp,lead_time_hours,forecast_issued_at,forecasted_temp_c,forecasted_precip_mm,forecasted_wind_kmh,forecasted_condition,forecasted_umbrella,forecasted_clothing,actual_temp_c,actual_precip_mm,actual_wind_kmh,actual_condition,actual_umbrella,actual_clothing,temp_error_c,precip_error_mm,wind_error_kmh,umbrella_correct,clothing_correct,condition_correct
0,FCST-000001,WS-001,2025-03-14 00:00:00,3,2025-03-13 21:00:00,-11.3,0.0,12.6,clear,0,heavy_winter_coat_gloves_hat,-12.1,0.0,12.0,clear,0,heavy_winter_coat_gloves_hat,0.8,0.0,0.6,1,1,1
1,FCST-000002,WS-001,2025-03-14 01:00:00,12,2025-03-13 13:00:00,-16.4,0.0,13.6,clear,0,heavy_winter_coat_gloves_hat,-12.5,0.0,4.7,clear,0,heavy_winter_coat_gloves_hat,-3.9,0.0,8.9,1,1,1
2,FCST-000003,WS-001,2025-03-14 02:00:00,1,2025-03-14 01:00:00,-11.0,0.0,4.2,clear,0,heavy_winter_coat_gloves_hat,-11.2,0.0,4.6,clear,0,heavy_winter_coat_gloves_hat,0.2,0.0,-0.4,1,1,1
3,FCST-000004,WS-001,2025-03-14 03:00:00,24,2025-03-13 03:00:00,-6.1,0.9,0.0,fog,0,light_jacket_or_sweater,-9.7,0.0,22.1,clear,0,heavy_winter_coat_gloves_hat,3.6,0.9,-22.1,1,0,0
4,FCST-000005,WS-001,2025-03-14 04:00:00,6,2025-03-13 22:00:00,-8.5,0.5,1.8,clear,0,winter_coat_scarf_gloves,-7.7,0.0,2.6,clear,0,winter_coat_scarf_gloves,-0.8,0.5,-0.8,1,1,1


In [3]:
df.dtypes

forecast_id                 str
station_id                  str
target_timestamp            str
lead_time_hours           int64
forecast_issued_at          str
forecasted_temp_c       float64
forecasted_precip_mm    float64
forecasted_wind_kmh     float64
forecasted_condition        str
forecasted_umbrella       int64
forecasted_clothing         str
actual_temp_c           float64
actual_precip_mm        float64
actual_wind_kmh         float64
actual_condition            str
actual_umbrella           int64
actual_clothing             str
temp_error_c            float64
precip_error_mm         float64
wind_error_kmh          float64
umbrella_correct          int64
clothing_correct          int64
condition_correct         int64
dtype: object

In [4]:
data = df.copy()

data["target_timestamp"] = pd.to_datetime(data["target_timestamp"])
data["forecast_issued_at"] = pd.to_datetime(data["forecast_issued_at"])

data["target_hour"] = data["target_timestamp"].dt.hour
data["target_month"] = data["target_timestamp"].dt.month
data["issue_hour"] = data["forecast_issued_at"].dt.hour
data["issue_month"] = data["forecast_issued_at"].dt.month

# Basic engineered weather indicators
data["is_rainy_forecast"] = (data["forecasted_precip_mm"] > 0.0).astype(int)
data["is_windy_forecast"] = (data["forecasted_wind_kmh"] >= 20.0).astype(int)
data["is_cold"] = (data["forecasted_temp_c"] <= 12.0).astype(int)
data["is_hot"] = (data["forecasted_temp_c"] >= 28.0).astype(int)

# Interaction features
data["precip_wind_interaction"] = data["forecasted_precip_mm"] * data["forecasted_wind_kmh"]
data["temp_wind_interaction"] = data["forecasted_temp_c"] * data["forecasted_wind_kmh"]

# Lead time as bucket
data["lead_time_bucket"] = pd.cut(
    data["lead_time_hours"],
    bins=[0, 1, 3, 6, 12, 24],
    labels=["h1", "h3", "h6", "h12", "h24"],
    include_lowest=True,
)

# Time of day buckets
def hour_bucket(hour):
    if 5 <= hour < 11:
        return "morning"
    elif 11 <= hour < 17:
        return "midday"
    elif 17 <= hour < 22:
        return "evening"
    else:
        return "night"

data["target_period"] = data["target_hour"].apply(hour_bucket)
data["issue_period"] = data["issue_hour"].apply(hour_bucket)

data.head()

,forecast_id,station_id,target_timestamp,lead_time_hours,forecast_issued_at,forecasted_temp_c,forecasted_precip_mm,forecasted_wind_kmh,forecasted_condition,forecasted_umbrella,forecasted_clothing,actual_temp_c,actual_precip_mm,actual_wind_kmh,actual_condition,actual_umbrella,actual_clothing,temp_error_c,precip_error_mm,wind_error_kmh,umbrella_correct,clothing_correct,condition_correct,target_hour,target_month,issue_hour,issue_month,is_rainy_forecast,is_windy_forecast,is_cold,is_hot,precip_wind_interaction,temp_wind_interaction,lead_time_bucket,target_period,issue_period
0,FCST-000001,WS-001,2025-03-14 00:00:00,3,2025-03-13 21:00:00,-11.3,0.0,12.6,clear,0,heavy_winter_coat_gloves_hat,-12.1,0.0,12.0,clear,0,heavy_winter_coat_gloves_hat,0.8,0.0,0.6,1,1,1,0,3,21,3,0,0,1,0,0.0,-142.38,h3,night,evening
1,FCST-000002,WS-001,2025-03-14 01:00:00,12,2025-03-13 13:00:00,-16.4,0.0,13.6,clear,0,heavy_winter_coat_gloves_hat,-12.5,0.0,4.7,clear,0,heavy_winter_coat_gloves_hat,-3.9,0.0,8.9,1,1,1,1,3,13,3,0,0,1,0,0.0,-223.04,h12,night,midday
2,FCST-000003,WS-001,2025-03-14 02:00:00,1,2025-03-14 01:00:00,-11.0,0.0,4.2,clear,0,heavy_winter_coat_gloves_hat,-11.2,0.0,4.6,clear,0,heavy_winter_coat_gloves_hat,0.2,0.0,-0.4,1,1,1,2,3,1,3,0,0,1,0,0.0,-46.20,h1,night,night
3,FCST-000004,WS-001,2025-03-14 03:00:00,24,2025-03-13 03:00:00,-6.1,0.9,0.0,fog,0,light_jacket_or_sweater,-9.7,0.0,22.1,clear,0,heavy_winter_coat_gloves_hat,3.6,0.9,-22.1,1,0,0,3,3,3,3,1,0,1,0,0.0,-0.00,h24,night,night
4,FCST-000005,WS-001,2025-03-14 04:00:00,6,2025-03-13 22:00:00,-8.5,0.5,1.8,clear,0,winter_coat_scarf_gloves,-7.7,0.0,2.6,clear,0,winter_coat_scarf_gloves,-0.8,0.5,-0.8,1,1,1,4,3,22,3,1,0,1,0,0.9,-15.30,h6,night,night


In [5]:
#Modeli eğitirken sadece o anda elimizde bulunabilecek feature’ları kullanacağız. Yani forecast var, actual yok
feature_cols = [
    "lead_time_hours",
    "forecasted_temp_c",
    "forecasted_precip_mm",
    "forecasted_wind_kmh",
    "forecasted_umbrella",
    "forecasted_clothing",
    "target_hour",
    "target_month",
    "issue_hour",
    "issue_month",
    "is_rainy_forecast",
    "is_windy_forecast",
    "is_cold",
    "is_hot",
    "precip_wind_interaction",
    "temp_wind_interaction",
    "lead_time_bucket",
    "target_period",
    "issue_period",
]

categorical_features = [
    "forecasted_clothing",
    "lead_time_bucket",
    "target_period",
    "issue_period",
]

numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X = data[feature_cols]
X.head()



,lead_time_hours,forecasted_temp_c,forecasted_precip_mm,forecasted_wind_kmh,forecasted_umbrella,forecasted_clothing,target_hour,target_month,issue_hour,issue_month,is_rainy_forecast,is_windy_forecast,is_cold,is_hot,precip_wind_interaction,temp_wind_interaction,lead_time_bucket,target_period,issue_period
0,3,-11.3,0.0,12.6,0,heavy_winter_coat_gloves_hat,0,3,21,3,0,0,1,0,0.0,-142.38,h3,night,evening
1,12,-16.4,0.0,13.6,0,heavy_winter_coat_gloves_hat,1,3,13,3,0,0,1,0,0.0,-223.04,h12,night,midday
2,1,-11.0,0.0,4.2,0,heavy_winter_coat_gloves_hat,2,3,1,3,0,0,1,0,0.0,-46.20,h1,night,night
3,24,-6.1,0.9,0.0,0,light_jacket_or_sweater,3,3,3,3,1,0,1,0,0.0,-0.00,h24,night,night
4,6,-8.5,0.5,1.8,0,winter_coat_scarf_gloves,4,3,22,3,1,0,1,0,0.9,-15.30,h6,night,night


In [6]:
for col in ["umbrella_correct", "clothing_correct", "condition_correct"]:
    print(df[col].value_counts(normalize=True).sort_index())

print("\nlead_time_hours distribution")
print(df["lead_time_hours"].value_counts().sort_index())

umbrella_correct
0    0.058125
1    0.941875
Name: proportion, dtype: float64
clothing_correct
0    0.271875
1    0.728125
Name: proportion, dtype: float64
condition_correct
0    0.141458
1    0.858542
Name: proportion, dtype: float64

lead_time_hours distribution
lead_time_hours
1     1009
3      901
6     1009
12     908
24     973
Name: count, dtype: int64


In [7]:
from xgboost import XGBClassifier
def train_confidence_model_xgb(dataframe, feature_cols, target_col, threshold=0.5):
    X = dataframe[feature_cols]
    y = dataframe[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    # imbalance için scale_pos_weight
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

    base_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.5,
        reg_lambda=2.0,
        gamma=0.2,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        )),
    ])

    base_model.fit(X_train, y_train)

    proba = base_model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)

    metrics = {
        "target": target_col,
        "base_rate": float(y.mean()),
        "accuracy": float(accuracy_score(y_test, pred)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "brier": float(brier_score_loss(y_test, proba)),
    }

    return base_model, metrics, X_test, y_test, proba



In [8]:
targets = ["umbrella_correct", "clothing_correct", "condition_correct"]

xgb_task_models = {}
xgb_task_metrics = []
xgb_task_test_outputs = {}

for target in targets:
    model, metrics, X_test, y_test, proba = train_confidence_model_xgb(
        data, feature_cols, target
    )
    xgb_task_models[target] = model
    xgb_task_metrics.append(metrics)
    xgb_task_test_outputs[target] = {
        "X_test": X_test,
        "y_test": y_test,
        "proba": proba,
    }

xgb_metrics_df = pd.DataFrame(xgb_task_metrics).sort_values("roc_auc", ascending=False)
xgb_metrics_df

,target,base_rate,accuracy,roc_auc,brier
0,umbrella_correct,0.941875,0.841667,0.925944,0.103289
2,condition_correct,0.858542,0.758333,0.906919,0.131805
1,clothing_correct,0.728125,0.742708,0.819622,0.175647


In [9]:
def normalize_confidence(score, lead_time, task):
    score = 0.5 + (score - 0.5) * 0.6

    if task == "umbrella":
        penalty = 1 - 0.004 * lead_time
    elif task == "clothing":
        penalty = 1 - 0.008 * lead_time
    else:  # condition
        penalty = 1 - 0.008 * lead_time

    score *= penalty
    score = max(0.05, min(0.95, score))
    return round(score, 4)

In [10]:
def predict_confidence_xgb(forecast_input: dict):
    sample = pd.DataFrame([forecast_input])
    lead_time = int(forecast_input["lead_time_hours"])
    

    umbrella_raw = float(xgb_task_models["umbrella_correct"].predict_proba(sample)[0, 1])
    clothing_raw = float(xgb_task_models["clothing_correct"].predict_proba(sample)[0, 1])
    condition_raw = float(xgb_task_models["condition_correct"].predict_proba(sample)[0, 1])

    umbrella_conf = normalize_confidence(umbrella_raw, lead_time, "umbrella")
    clothing_conf = normalize_confidence(clothing_raw, lead_time, "clothing")
    condition_conf = normalize_confidence(condition_raw, lead_time, "condition")

    return {
        "umbrella_confidence": umbrella_conf,
        "clothing_confidence": clothing_conf,
        "condition_confidence": condition_conf,
    }


In [11]:
def confidence_label(score: float) -> str:
    if score >= 0.75:
        return "high"
    elif score >= 0.50:
        return "medium"
    else:
        return "low"

In [12]:
def enrich_confidence_output(pred: dict) -> dict:
    enriched = {}
    for key, value in pred.items():
        enriched[key] = {
            "score": round(float(value), 4),
            "label": confidence_label(float(value)),
        }
    return enriched

In [13]:
sample1 = {
    "lead_time_hours": 1,
    "forecasted_temp_c": 30.0,
    "forecasted_precip_mm": 0.0,
    "forecasted_wind_kmh": 8.0,
    "forecasted_umbrella": 0,
    "forecasted_clothing": "very_light_clothing",
    "target_hour": 14,
    "target_month": 7,
    "issue_hour": 13,
    "issue_month": 7,
    "is_rainy_forecast": 0,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 1,
    "precip_wind_interaction": 0.0,
    "temp_wind_interaction": 240.0,
    "lead_time_bucket": "h1",
    "target_period": "midday",
    "issue_period": "midday",
}


sample2 = {
    "lead_time_hours": 24,
    "forecasted_temp_c": 28.0,
    "forecasted_precip_mm": 0.0,
    "forecasted_wind_kmh": 10.0,
    "forecasted_umbrella": 0,
    "forecasted_clothing": "very_light_clothing",
    "target_hour": 10,
    "target_month": 7,
    "issue_hour": 10,
    "issue_month": 7,
    "is_rainy_forecast": 0,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 1,
    "precip_wind_interaction": 0.0,
    "temp_wind_interaction": 280.0,
    "lead_time_bucket": "h24",
    "target_period": "midday",
    "issue_period": "midday",
}

sample3 = {
    "lead_time_hours": 12,
    "forecasted_temp_c": 11.0,
    "forecasted_precip_mm": 5.0,
    "forecasted_wind_kmh": 32.0,
    "forecasted_umbrella": 1,
    "forecasted_clothing": "heavy_winter_coat_gloves_hat",
    "target_hour": 9,
    "target_month": 1,
    "issue_hour": 21,
    "issue_month": 12,
    "is_rainy_forecast": 1,
    "is_windy_forecast": 1,
    "is_cold": 1,
    "is_hot": 0,
    "precip_wind_interaction": 160.0,
    "temp_wind_interaction": 352.0,
    "lead_time_bucket": "h12",
    "target_period": "morning",
    "issue_period": "night",
}

sample4 = {
    "lead_time_hours": 6,
    "forecasted_temp_c": 20.0,
    "forecasted_precip_mm": 0.2,
    "forecasted_wind_kmh": 18.0,
    "forecasted_umbrella": 0,
    "forecasted_clothing": "light_jacket_or_sweater",
    "target_hour": 19,
    "target_month": 5,
    "issue_hour": 13,
    "issue_month": 5,
    "is_rainy_forecast": 1,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 0,
    "precip_wind_interaction": 3.6,
    "temp_wind_interaction": 360.0,
    "lead_time_bucket": "h6",
    "target_period": "evening",
    "issue_period": "midday",
}

sample5 = {
    "lead_time_hours": 3,
    "forecasted_temp_c": 16.0,
    "forecasted_precip_mm": 1.5,
    "forecasted_wind_kmh": 12.0,
    "forecasted_umbrella": 1,
    "forecasted_clothing": "long_sleeves_light_layer",
    "target_hour": 23,
    "target_month": 10,
    "issue_hour": 20,
    "issue_month": 10,
    "is_rainy_forecast": 1,
    "is_windy_forecast": 0,
    "is_cold": 0,
    "is_hot": 0,
    "precip_wind_interaction": 18.0,
    "temp_wind_interaction": 192.0,
    "lead_time_bucket": "h3",
    "target_period": "night",
    "issue_period": "evening",
}


samples = [sample1, sample2, sample3, sample4, sample5]

for i, sample in enumerate(samples, start=1):
    raw_pred = predict_confidence_xgb(sample)
    enriched = enrich_confidence_output(raw_pred)
    print(f"--- Sample {i} ---")
    print(enriched)
    print()

--- Sample 1 ---
{'umbrella_confidence': {'score': 0.7741, 'label': 'high'}, 'clothing_confidence': {'score': 0.72, 'label': 'medium'}, 'condition_confidence': {'score': 0.7851, 'label': 'high'}}

--- Sample 2 ---
{'umbrella_confidence': {'score': 0.2983, 'label': 'low'}, 'clothing_confidence': {'score': 0.2761, 'label': 'low'}, 'condition_confidence': {'score': 0.2286, 'label': 'low'}}

--- Sample 3 ---
{'umbrella_confidence': {'score': 0.3213, 'label': 'low'}, 'clothing_confidence': {'score': 0.4173, 'label': 'low'}, 'condition_confidence': {'score': 0.3463, 'label': 'low'}}

--- Sample 4 ---
{'umbrella_confidence': {'score': 0.7526, 'label': 'high'}, 'clothing_confidence': {'score': 0.5993, 'label': 'medium'}, 'condition_confidence': {'score': 0.7374, 'label': 'medium'}}

--- Sample 5 ---
{'umbrella_confidence': {'score': 0.7551, 'label': 'high'}, 'clothing_confidence': {'score': 0.6373, 'label': 'medium'}, 'condition_confidence': {'score': 0.768, 'label': 'high'}}



In [14]:
import joblib

joblib.dump(xgb_task_models["umbrella_correct"], "../../models/umbrella_confidence_model.pkl")
joblib.dump(xgb_task_models["clothing_correct"], "../../models/clothing_confidence_model.pkl")
joblib.dump(xgb_task_models["condition_correct"], "../../models/condition_confidence_model.pkl")

['../../models/condition_confidence_model.pkl']